In [ ]:
import pandas as pd
import xml.etree.ElementTree as ET
from sklearn.preprocessing import MinMaxScaler
from tqdm import tqdm

## Data

### PPI

In [ ]:
protein_interaction = pd.read_csv('Data/Protein-protein interaction data/9606.protein.links.v12.0.txt', sep= ' ')
protein_interaction_full = pd.read_csv('Data/Protein-protein interaction data/9606.protein.links.full.v12.0.txt', sep= ' ')
protein_interaction_detailed = pd.read_csv('Data/Protein-protein interaction data/9606.protein.links.detailed.v12.0.txt', sep= ' ')
### convert proteins to their true names
protein_info = pd.read_csv('Data/Protein-protein interaction data/9606.protein.info.v12.0.txt', on_bad_lines='skip', sep='\t')
protein_aliases= pd.read_csv('Data/Protein-protein interaction data/9606.protein.aliases.v12.0.txt', on_bad_lines='skip', sep='\t')

In [ ]:
### convert proteins to their true names
protein_info = pd.read_csv('Data/Protein-protein interaction data/9606.protein.info.v12.0.txt', on_bad_lines='skip', sep='\t')
protein_aliases= pd.read_csv('Data/Protein-protein interaction data/9606.protein.aliases.v12.0.txt', on_bad_lines='skip', sep='\t')

# Method 1: Using the to_dict() method with 'index' as orient
protein_info_translate_name_dict = protein_info.set_index('#string_protein_id')['preferred_name'].to_dict()
protein_alias_translate_name_dict = protein_aliases.set_index('#string_protein_id')['alias'].to_dict()
#print(protein_info_translate_name_dict)

### Protein1
protein1_name = []
for prot_id in tqdm(protein_interaction['protein1']):
    if prot_id in protein_info_translate_name_dict:
        protein1_name.append(protein_info_translate_name_dict[prot_id])
    elif prot_id in protein_alias_translate_name_dict:
        protein1_name.append(protein_alias_translate_name_dict[prot_id])
    else:
        protein1_name.append('')

### Protein 2
protein2_name = []
for prot_id in tqdm(protein_interaction['protein2']):
    if prot_id in protein_info_translate_name_dict:
        protein2_name.append(protein_info_translate_name_dict[prot_id])
    elif prot_id in protein_alias_translate_name_dict:
        protein2_name.append(protein_alias_translate_name_dict[prot_id])
    else:
        protein2_name.append('')

protein_interaction['Translated_protein_1'] = protein1_name
protein_interaction['Translated_protein_2'] = protein2_name

# Create a set of all (protein1, protein2) pairs
ppi_pairs = set(zip(protein_interaction['Translated_protein_1'], protein_interaction['Translated_protein_2']))
# Check for missing reverse pairs
missing_reverse = []
for a, b in ppi_pairs:
    if (b, a) not in ppi_pairs:
        missing_reverse.append((a, b))

print(f"Number of pairs missing their reverse: {len(missing_reverse)}")
if missing_reverse:
    print("Examples:", missing_reverse[:10])
else:
    print("All pairs have their reverse present.")

In [ ]:
protein_interaction

### DrugBank

In [ ]:
# import xml.etree.ElementTree as ET

# # Load XML
# drugbank_xml = 'Data/DGIDB/drug_bank.xml'
# tree = ET.parse(drugbank_xml)
# root = tree.getroot()

# # Namespace
# ns = {'db': 'http://www.drugbank.ca'}

# Helper to clean tag names
def clean_tag(tag):
    return tag.split('}')[-1] if '}' in tag else tag

# Recursive function to print structure
def print_structure(elem, level=0):
    indent = '  ' * level
    print(f"{indent}- {clean_tag(elem.tag)}")
    for child in elem:
        print_structure(child, level + 1)

# # Get first drug
# first_drug = root.find('db:drug', ns)

# print("🌿 Structure of First Drug Entry:")
# print_structure(first_drug)
# print("\n🌳 Structure of First 3 Drug Entries:")
# drugs = root.findall('db:drug', ns)

# for i, drug in enumerate(drugs[:3]):
#     print(f"\n🔬 Drug {i+1}:")
#     print_structure(drug)


In [ ]:
def structure_drug_bank_data(drug_bank_file = 'Data/DGIDB/drug_bank.xml'):
    """
    Function to structure the drug bank data from the XML file.
    :param drug_bank_file: Path to the drug bank XML file.
    :return: DataFrame containing structured drug bank data.
    """
    ### FYI the .find command only finds the first instance of a tag, 
    ### while .findall retrieves all instances of the specified tag within the current element.

    tree = ET.parse(drug_bank_file)
    root = tree.getroot()

    # DrugBank uses a specific namespace
    ns = {'db': 'http://www.drugbank.ca'}
    ### extract all drug elements
    drugs = root.findall('db:drug', ns)
    print(f"Found {len(drugs)} drugs in the DrugBank XML.")
    # Extract drug-gene interactions
    interactions = []
    # The interactions list will store dictionaries with 'drug' and 'gene' keys.
    for drug in root.findall('db:drug', ns): # root.findall('db:drug', ns): Finds all <drug> elements using the namespace.
        drug_name  = drug.find('db:name', ns).text  # drug.find('db:name', ns): Gets the drug's name.
        # print(drug_name)
        for target in drug.findall('db:targets/db:target', ns):  # drug.findall('db:targets/db:target', ns): Finds all <target> elements within <targets>.
            # print(target.tag)
            gene_description = target.find('db:name', ns)  # target.find('db:name', ns): Extracts the gene name for each target.
            poly = target.find('db:polypeptide', ns)  # target.find('db:polypeptide', ns): Extracts the polypeptide information.
            action = target.find('db:actions/db:action', ns) # target.find('db:actions/db:action', ns): Extracts the action of the drug on the target.
            if poly is not None:
                poly_name = poly.find('db:name', ns)
                gene_name = poly.find('db:gene-name', ns)
                specific_function = poly.find('db:specific-function', ns)
                interactions.append({
                    'drug': drug_name,
                    'polypeptide': poly_name.text if poly_name is not None else None,
                    'gene': gene_name.text if gene_name is not None else None,
                    'gene_description': gene_description.text if gene_description is not None else None,
                    'action': action.text if action is not None else None,
                    'specific_function': specific_function.text if specific_function is not None else None
                })
            ############# if polypeptide is not present, we still want to add the drug and gene information
            ############# this is because some drugs may not have a polypeptide associated with them
            ############# but we still want to capture the drug and gene information
            ############# this is common in the DrugBank database, where some drugs target genes directly
            ############# and do not have a polypeptide associated with them

            else:
                gene_name = None
                specific_function = None
                poly_name = None
                action = None
                gene_description = None
                resource = None
                identifier = None
  
                interactions.append({
                        'drug': drug_name,
                        'polypeptide': poly_name.text if poly_name is not None else None,
                        'gene': gene_name.text if gene_name is not None else None,
                        'gene_description': gene_description.text if gene_description is not None else None,
                        'action': action.text if action is not None else None,
                        'specific_function': specific_function.text if specific_function is not None else None
                    })
        
    # Convert to DataFrame
    # Converts the list of dictionaries into a pandas DataFrame, which is easier to analyze, filter, and export.
    df = pd.DataFrame(interactions)

    return df

In [ ]:
Drug_bank = structure_drug_bank_data('Data/DGIDB/drug_bank.xml')

In [ ]:
Drug_bank['drug'].unique()
print(f"Total unique drugs: {len(Drug_bank['gene'].unique())}")

In [ ]:
Drug_bank_temp = Drug_bank[~Drug_bank['gene'].isnull()]
Drug_bank_temp['gene'].nunique()

### Genetic results

In [ ]:
### import data

### genes
hpv_positive_genes  = pd.read_csv('Results/CNV results/HPV positive CNV top genes.csv')
hpv_negative_genes = pd.read_csv('Results/CNV results/HPV negative CNV top genes.csv')

### drug candiates
hpv_positive_direct_drug_candidates = pd.read_csv('Results/CNV results/HPV Positive Top Direct Drug Candidates Aggregated.csv')
hpv_positive_indirect_drug_candidates = pd.read_csv('Results/CNV results/HPV Positive Top Indirect Drug Candidates Aggregated.csv')

hpv_negative_direct_drug_candidates = pd.read_csv('Results/CNV results/HPV Negative Top Direct Drug Candidates Aggregated.csv')
#### no direct drug candidates came from Deletions, only amplifications
hpv_negative_direct_drug_candidates['MUT_TYPE'] = 'AMPLIFICATION'
hpv_negative_indirect_drug_candidates = pd.read_csv('Results/CNV results/HPV Negative Top Indirect Drug Candidates Aggregated.csv')

### somatic mtuation
hpv_positive_som_genes = pd.read_csv('Results/SOM results/HPV positive top genes.csv')
hpv_positive_som_direct_drug_candidates = pd.read_csv('Results/SOM results/hpv_positive_som_top_direct_drug_candidates_agg.csv')
hpv_positive_som_direct_drug_candidates['MUT_TYPE'] = 'SOMATIC'
hpv_positive_som_indirect_drug_candidates = pd.read_csv('Results/SOM results/hpv_positive_som_top_indirect_drug_candidates_agg.csv')
hpv_positive_som_indirect_drug_candidates['MUT_TYPE'] = 'SOMATIC'

hpv_negative_som_genes = pd.read_csv('Results/SOM results/HPV negative top genes.csv')
hpv_negative_som_direct_drug_candidates = pd.read_csv('Results/SOM results/hpv_negative_som_top_direct_drug_candidates_agg.csv')
hpv_negative_som_direct_drug_candidates['MUT_TYPE'] = 'SOMATIC'
hpv_negative_som_indirect_drug_candidates = pd.read_csv('Results/SOM results/hpv_negative_som_top_indirect_drug_candidates_agg.csv')
hpv_negative_som_indirect_drug_candidates['MUT_TYPE'] = 'SOMATIC'

#### Overlap

##### HPV+

In [ ]:
### number of unique drugs of all hpv positive both direct and indirect
num_unique_pos_drugs = len(list(set(list(set(hpv_positive_direct_drug_candidates['DRUG'].str.lower()))
                           + list(set(hpv_positive_indirect_drug_candidates['DRUG'].str.lower())) 
                           + list(set(hpv_positive_som_direct_drug_candidates['DRUG'].str.lower())) 
                           + list(set(hpv_positive_som_indirect_drug_candidates['DRUG'].str.lower())))))
num_unique_pos_drugs

In [ ]:
# Get the set of unique HPV positive direct drug candidates
hpv_positive_direct_drugs = set(hpv_positive_direct_drug_candidates['DRUG'].str.lower())
hpv_positive_som_direct_drugs = set(hpv_positive_som_direct_drug_candidates['DRUG'].str.lower())

# Combine both sets to get all unique HPV positive direct drug candidates
all_hpv_positive_direct_drugs = hpv_positive_direct_drugs.union(hpv_positive_som_direct_drugs)

print(f"Number of unique HPV positive direct drug candidates: {len(all_hpv_positive_direct_drugs)}")
print("\nHPV positive direct drug candidates:")
print(sorted(all_hpv_positive_direct_drugs))

In [ ]:
# Get the set of unique HPV positive indirect drug candidates
hpv_positive_indirect_drugs = set(hpv_positive_indirect_drug_candidates['DRUG'].str.lower())
hpv_positive_som_indirect_drugs = set(hpv_positive_som_indirect_drug_candidates['DRUG'].str.lower())

# Combine both sets to get all unique HPV positive indirect drug candidates
all_hpv_positive_indirect_drugs = hpv_positive_indirect_drugs.union(hpv_positive_som_indirect_drugs)

print(f"Number of unique HPV positive indirect drug candidates: {len(all_hpv_positive_indirect_drugs)}")
print("\nHPV positive indirect drug candidates:")
print(sorted(all_hpv_positive_indirect_drugs))

In [ ]:
# Get the set of unique HPV positive direct drug candidates
hpv_positive_direct_drugs = set(hpv_positive_direct_drug_candidates['DRUG'].str.lower())
hpv_positive_som_direct_drugs = set(hpv_positive_som_direct_drug_candidates['DRUG'].str.lower())

# Get the set of unique HPV positive indirect drug candidates
hpv_positive_indirect_drugs = set(hpv_positive_indirect_drug_candidates['DRUG'].str.lower())
hpv_positive_som_indirect_drugs = set(hpv_positive_som_indirect_drug_candidates['DRUG'].str.lower())

# Combine both sets to get all unique HPV positive direct and indirect drug candidates
all_hpv_positive_direct_drugs = hpv_positive_direct_drugs.union(hpv_positive_som_direct_drugs)
all_hpv_positive_indirect_drugs = hpv_positive_indirect_drugs.union(hpv_positive_som_indirect_drugs)


# Find the overlap between direct and indirect drug candidates
overlapping_drugs = all_hpv_positive_direct_drugs.intersection(all_hpv_positive_indirect_drugs)

print(f"Number of unique HPV positive direct drug candidates: {len(all_hpv_positive_direct_drugs)}")
print(f"Number of unique HPV positive indirect drug candidates: {len(all_hpv_positive_indirect_drugs)}")
print(f"Number of overlapping drugs between direct and indirect: {len(overlapping_drugs)}")
print("\nOverlapping drugs:")
print(sorted(overlapping_drugs))

##### HPV-

In [ ]:
### number of unique drugs of all hpv negative both direct and indirect
num_unique_neg_drugs = len(list(set(list(set(hpv_negative_direct_drug_candidates['DRUG'].str.lower()))
                           + list(set(hpv_negative_indirect_drug_candidates['DRUG'].str.lower())) 
                           + list(set(hpv_negative_som_direct_drug_candidates['DRUG'].str.lower())) 
                           + list(set(hpv_negative_som_indirect_drug_candidates['DRUG'].str.lower())))))
num_unique_neg_drugs

In [ ]:
## HPV- Drug Candidates
### Number of unique direct drug candidates
# Get the set of unique HPV negative direct drug candidates
hpv_negative_direct_drugs = set(hpv_negative_direct_drug_candidates['DRUG'].str.lower())
hpv_negative_som_direct_drugs = set(hpv_negative_som_direct_drug_candidates['DRUG'].str.lower())

# Combine both sets to get all unique HPV negative direct drug candidates
all_hpv_negative_direct_drugs = hpv_negative_direct_drugs.union(hpv_negative_som_direct_drugs)

print(f"Number of unique HPV negative direct drug candidates: {len(all_hpv_negative_direct_drugs)}")
print("\nHPV negative direct drug candidates:")
print(sorted(all_hpv_negative_direct_drugs))


In [ ]:
# Get the set of unique HPV negative indirect drug candidates
hpv_negative_indirect_drugs = set(hpv_negative_indirect_drug_candidates['DRUG'].str.lower())
hpv_negative_som_indirect_drugs = set(hpv_negative_som_indirect_drug_candidates['DRUG'].str.lower())

# Combine both sets to get all unique HPV negative indirect drug candidates
all_hpv_negative_indirect_drugs = hpv_negative_indirect_drugs.union(hpv_negative_som_indirect_drugs)

print(f"Number of unique HPV negative indirect drug candidates: {len(all_hpv_negative_indirect_drugs)}")
print("\nHPV negative indirect drug candidates:")
print(sorted(all_hpv_negative_indirect_drugs))

In [ ]:
# Get the set of unique HPV negative drug candidates (both direct and indirect)
hpv_negative_direct_drugs = set(hpv_negative_direct_drug_candidates['DRUG'].str.lower())
hpv_negative_som_direct_drugs = set(hpv_negative_som_direct_drug_candidates['DRUG'].str.lower())
hpv_negative_indirect_drugs = set(hpv_negative_indirect_drug_candidates['DRUG'].str.lower())
hpv_negative_som_indirect_drugs = set(hpv_negative_som_indirect_drug_candidates['DRUG'].str.lower())

# Combine all HPV negative drug sets
all_hpv_negative_direct_drugs = hpv_negative_direct_drugs.union(hpv_negative_som_direct_drugs)
all_hpv_negative_direct_drugs = set(drug.lower() for drug in all_hpv_negative_direct_drugs)
all_hpv_negative_indirect_drugs = hpv_negative_indirect_drugs.union(hpv_negative_som_indirect_drugs)
all_hpv_negative_indirect_drugs = set(drug.lower() for drug in all_hpv_negative_indirect_drugs)

# Find the overlap between direct and indirect drug candidates for HPV negative
overlapping_drugs = all_hpv_negative_direct_drugs.intersection(all_hpv_negative_indirect_drugs)


print(f"Number of unique HPV negative direct drug candidates: {len(all_hpv_negative_direct_drugs)}")
print(f"Number of unique HPV negative indirect drug candidates: {len(all_hpv_negative_indirect_drugs)}")
print(f"Number of overlapping drugs between direct and indirect: {len(overlapping_drugs)}")
print("\nOverlapping drugs:")
print(sorted(overlapping_drugs))

#### Literature results

In [ ]:
extracted_target_df= pd.read_csv('Validation pipeline/Results/cleaned_extracted_targets_all_pub_after_2000_GPU_2b_gemma.csv')
extracted_target_df_combined = pd.read_csv('Validation pipeline/Results/cleaned_extracted_combined_targets_all_pub_after_2000_GPU_2b_gemma.csv')

In [ ]:
### accumulate all genes available in drugbank or ppi
Drug_bank_genes = list(Drug_bank['gene'].values)
ppi_genes = list(protein_interaction['Translated_protein_1'].values)
all_ppi_drugbank = list(set(Drug_bank_genes + ppi_genes))

## HPV+

#### Genes

In [ ]:
hpv_positive_som_genes

In [ ]:
### combine hpv positive somatic genes and cnv genes
hpv_positive_som_genes['MUT_TYPE'] = 'SOMATIC'
hpv_positive_som_genes['gene_name'] = hpv_positive_som_genes['Gene']
hpv_positive_som_genes['q_value']= hpv_positive_som_genes['Adjusted_P_Value']
hpv_positive_som_genes['empirical_q_value'] = hpv_positive_som_genes['Adjusted_Empirical_P_Value']
hpv_positive_combined_genes = pd.concat([hpv_positive_genes, hpv_positive_som_genes], axis=0)
### aggregate by GENE to get unique genes with both mutation types
hpv_positive_combined_genes['gene_name'] = hpv_positive_combined_genes['gene_name'].str.upper()
extracted_target_df_combined['GENE'] = extracted_target_df_combined['GENE'].str.upper()
hpv_positive_combined_genes = hpv_positive_combined_genes.groupby('gene_name').agg({
    'MUT_TYPE': lambda x: ', '.join(x),
    'q_value': lambda x: ', '.join(x.astype(str)) if len(x) > 1 else x.iloc[0].astype(str),
    'empirical_q_value': lambda x: ', '.join(x.astype(str)) if len(x) > 1 else x.iloc[0].astype(str)
}).reset_index()

hpv_positive_combined_genes


In [ ]:
len(set(hpv_positive_combined_genes['gene_name']))

In [ ]:
### merge genes with number of articles, pubmed id from literature data
hpv_positive_genes_with_lit = pd.merge(hpv_positive_combined_genes, extracted_target_df_combined, how = 'left', left_on='gene_name', right_on='GENE')
hpv_positive_genes_with_lit.drop(columns =['INDEX'], inplace = True)

In [ ]:
hpv_positive_genes_with_lit[hpv_positive_genes_with_lit['NUMBER_OF_ARTICLES']>0]

In [ ]:
hpv_positive_genes_with_lit = hpv_positive_genes_with_lit[hpv_positive_genes_with_lit['NUMBER_OF_ARTICLES']>0]

In [ ]:
hpv_positive_genes_with_lit.to_csv('Results/HPV positive gene results.csv')

In [ ]:
### export to final results output
hpv_positive_genes_with_lit.to_csv('Results/Final Results/HPV Positive validated genes.csv')

#### Direct

In [ ]:
### merge all hpv postive direct drug candidates
hpv_positive_final_direct = pd.concat([hpv_positive_direct_drug_candidates, hpv_positive_som_direct_drug_candidates])
### group by drug and comma seperate genes and mutation type
### columns: DRUG	GENE_TARGET	NUM_DIRECT_TARGETS_HIT	TOTAL_TARGETS_IN_DRUGBANK	PERCENTAGE_OF_TARGETS_HIT	GENE_GISTIC	GENE_normalized_gistic_score	
# ACTION	SPECIFIC_FUNCTION	drug_hypergeom_p_value	drug_hypergeom_fdr	drug_empirical_p_value	drug_empirical_fdr	MUT_TYPE	GENE_Cohort_Frequency	
# GENE_Normalized_Count	GENE_Normalized_Cohort_Frequency	GENE_SIGNIFICANT
hpv_positive_final_direct = hpv_positive_final_direct.groupby('DRUG').agg({'GENE_TARGET': lambda x: ', '.join(x),
                                               'MUT_TYPE': lambda x: ', '.join(x),
                                                  'NUM_DIRECT_TARGETS_HIT': 'first',
                                                    'TOTAL_TARGETS_IN_DRUGBANK': 'first',
                                                    'PERCENTAGE_OF_TARGETS_HIT': 'first',
                                                    'ACTION': 'first',
                                                    'SPECIFIC_FUNCTION': 'first',
                                                    'drug_hypergeom_p_value': 'first',
                                                    'drug_hypergeom_fdr': 'first',
                                                    'drug_empirical_p_value': 'first',
                                                    'drug_empirical_fdr': 'first',
                                                    }).reset_index()

In [ ]:
hpv_positive_final_direct

In [ ]:
extracted_target_df_combined

In [ ]:
#### add columns to hpv_positive_final_direct for PMIds and NUMBER_OF_ARTICLES from extracted_target_df_combined
### ADD COLUMNS: PMIDs, NUMBER_OF_ARTICLES, gene
### combine based on gene target, and if any of the genes in GENE_TARGET are in extracted_target_df_combined, then add the PMIDs and NUMBER_OF_ARTICLES

hpv_positive_final_direct['PMID'] = ''
hpv_positive_final_direct['NUMBER_OF_ARTICLES'] = 0
hpv_positive_final_direct['LITERATURE_GENE_TARGETS'] = ''
for index, row in hpv_positive_final_direct.iterrows():
    gene_targets = row['GENE_TARGET'].split(', ')
    gene_targets = [gene.strip() for gene in gene_targets]
    gene_targets = list(set(gene_targets))
    pmids_set = set()
    literature_gene_targets = set()
    number_of_articles = 0
    for gene in gene_targets:
        matched_rows = extracted_target_df_combined[extracted_target_df_combined['GENE'] == gene]
        for _, matched_row in matched_rows.iterrows():
            pmids = matched_row['PMID'].split(', ')
            pmids_set.update(pmids)
            number_of_articles += matched_row['NUMBER_OF_ARTICLES']
            literature_gene_targets.add(matched_row['GENE'])
    hpv_positive_final_direct.at[index, 'LITERATURE_GENE_TARGETS'] = ', '.join(list(set(literature_gene_targets)))
    hpv_positive_final_direct.at[index, 'PMID'] = ', '.join(pmids_set)
    hpv_positive_final_direct.at[index, 'NUMBER_OF_ARTICLES'] = number_of_articles

In [ ]:
hpv_positive_final_direct[hpv_positive_final_direct['NUMBER_OF_ARTICLES'] > 0]

In [ ]:
### ensure that only drugs with NUMBER_OF_ARTICLES > 0 are saved, so that they have literature support
hpv_positive_final_direct = hpv_positive_final_direct[hpv_positive_final_direct['NUMBER_OF_ARTICLES'] > 0]
### save results
hpv_positive_final_direct.to_csv('Results/HPV Positive direct results.csv')

#### Indirect

In [ ]:
hpv_positive_indirect_drug_candidates[hpv_positive_indirect_drug_candidates['DRUG'].str.lower()== 'quercetin']

In [ ]:
### merge all hpv positive indirect drug candidates
hpv_positive_final_indirect = pd.concat([hpv_positive_indirect_drug_candidates, hpv_positive_som_indirect_drug_candidates], ignore_index=True)
hpv_positive_final_indirect['ACTION'] = hpv_positive_final_indirect['ACTION'].fillna('UNKNOWN')
hpv_positive_final_indirect['SPECIFIC_FUNCTION'] = hpv_positive_final_indirect['SPECIFIC_FUNCTION'].fillna('UNKNOWN')
hpv_positive_final_indirect['drug_hypergeom_fdr'] = hpv_positive_final_indirect['drug_hypergeom_fdr'].fillna('UNKNOWN')
hpv_positive_final_indirect['drug_empirical_fdr'] = hpv_positive_final_indirect['drug_empirical_fdr'].fillna('UNKNOWN')
hpv_positive_final_indirect['MUT_TYPE'] = hpv_positive_final_indirect['MUT_TYPE'].fillna('UNKNOWN')

### aggregate/group by drug name
### columns: DRUG	CONNECTED_TO (risk gene)	
# Number of risk or immediate neighbor genes
# targeted	total_genes_targeted_in_drugbank	
# PERCENTAGE_OF_TARGETS_HIT
# Number of indirect genes connected to this risk gene	
# GENE_TARGET	GENE_Cohort_Frequency	
# GENE_Normalized_Count	GENE_Normalized_Cohort_Frequency	
# ACTION	SPECIFIC_FUNCTION	drug_hypergeom_fdr	drug_empirical_fdr

hpv_positive_final_indirect = hpv_positive_final_indirect.groupby(['DRUG']).agg({
    'GENE_TARGET': lambda x: ', '.join(x),
    'CONNECTED_TO (risk gene)': lambda x: ', '.join(x),
    'Number of risk or immediate neighbor genes targeted': 'first',
    'total_genes_targeted_in_drugbank': 'first',
    'PERCENTAGE_OF_TARGETS_HIT': 'first',
    'ACTION': lambda x: ', '.join(x),
    'SPECIFIC_FUNCTION': lambda x: ', '.join(x),
    'drug_hypergeom_fdr': 'max',
    'drug_empirical_fdr': 'max',
    'MUT_TYPE': lambda x: ', '.join(x)
}).reset_index()
hpv_positive_final_indirect.sort_values(by = 'drug_empirical_fdr', ascending = True).head(25)

In [ ]:
# hpv_positive_final_indirect[hpv_positive_final_indirect['DRUG'].str.lower().isin(hpv_positive_final_direct['DRUG'].str.lower())]

In [ ]:
### extract the unique number of GENE_TARGETs from , seperated list in each of the GENE_TARGET column
def extract_unique_gene_targets(df, column_name):
    return len(set([gene.strip() for sublist in df[column_name].dropna().str.split(',') for gene in sublist]))

hpv_pos_indirect_genes = extract_unique_gene_targets(hpv_positive_indirect_drug_candidates, 'GENE_TARGET')
print(hpv_pos_indirect_genes)

In [ ]:
#### add columns to hpv_positive_final_direct for PMIds and NUMBER_OF_ARTICLES from extracted_target_df_combined
### ADD COLUMNS: PMIDs, NUMBER_OF_ARTICLES, gene
### combine based on gene target, and if any of the genes in GENE_TARGET are in extracted_target_df_combined, then add the PMIDs and NUMBER_OF_ARTICLES
hpv_positive_final_indirect['PMID'] = ''
hpv_positive_final_indirect['NUMBER_OF_ARTICLES'] = 0
hpv_positive_final_indirect['LITERATURE_GENE_TARGETS'] = ''
for index, row in hpv_positive_final_indirect.iterrows():
    gene_targets = row['GENE_TARGET'].split(',')
    gene_targets = [gene.strip() for gene in gene_targets]
    pmids_set = set()
    literature_gene_targets = set()
    number_of_articles = 0
    for gene in gene_targets:
        matched_rows = extracted_target_df_combined[extracted_target_df_combined['GENE'] == gene]
        for _, matched_row in matched_rows.iterrows():
            pmids = matched_row['PMID'].split(',')
            pmids = [pmid.strip() for pmid in pmids]
            pmids_set.update(pmids)
            number_of_articles += matched_row['NUMBER_OF_ARTICLES']
            literature_gene_targets.add(matched_row['GENE'])
    
    hpv_positive_final_indirect.at[index, 'LITERATURE_GENE_TARGETS'] = ', '.join(literature_gene_targets)
    hpv_positive_final_indirect.at[index, 'PMID'] = ', '.join(pmids_set)
    hpv_positive_final_indirect.at[index, 'NUMBER_OF_ARTICLES'] = number_of_articles

### validate risk genes
hpv_positive_final_indirect = hpv_positive_final_indirect[hpv_positive_final_indirect['NUMBER_OF_ARTICLES'] > 0]
hpv_positive_final_indirect['RISK_GENE_PMID'] = ''
hpv_positive_final_indirect['RISK_GENE_NUMBER_OF_ARTICLES'] = 0
hpv_positive_final_indirect['RISK_GENE_LITERATURE_GENE_TARGETS'] = ''
for index, row in hpv_positive_final_indirect.iterrows():
    risk_genes = row['CONNECTED_TO (risk gene)'].split(',')
    risk_genes = [gene.strip() for gene in risk_genes]
    pmids_set = set()
    literature_gene_targets = set()
    number_of_articles = 0
    for gene in risk_genes:
        #print(gene)
        matched_rows = extracted_target_df_combined[extracted_target_df_combined['GENE'] == gene]
        for _, matched_row in matched_rows.iterrows():
            pmids = matched_row['PMID'].split(',')
            pmids = [pmid.strip() for pmid in pmids]
            pmids_set.update(pmids)
            number_of_articles += matched_row['NUMBER_OF_ARTICLES']
            literature_gene_targets.add(matched_row['GENE'])
    
    hpv_positive_final_indirect.at[index, 'RISK_GENE_LITERATURE_GENE_TARGETS'] = ', '.join(literature_gene_targets)
    hpv_positive_final_indirect.at[index, 'RISK_GENE_PMID'] = ', '.join(pmids_set)
    hpv_positive_final_indirect.at[index, 'RISK_GENE_NUMBER_OF_ARTICLES'] = number_of_articles

### ensure that only drugs with NUMBER_OF_ARTICLES > 0 for both drug targets and risk genes are saved, so that they have literature support
hpv_positive_final_indirect = hpv_positive_final_indirect[hpv_positive_final_indirect['NUMBER_OF_ARTICLES'] > 0]
hpv_positive_final_indirect = hpv_positive_final_indirect[hpv_positive_final_indirect['RISK_GENE_NUMBER_OF_ARTICLES'] > 0]

In [ ]:
### extract the unique number of GENE_TARGETs from , seperated list in each of the GENE_TARGET column
def extract_unique_gene_targets(df, column_name):
    return len(set([gene.strip() for sublist in df[column_name].dropna().str.split(',') for gene in sublist]))

hpv_pos_final_indirect_genes = extract_unique_gene_targets(hpv_positive_final_indirect, 'GENE_TARGET')
print(hpv_pos_final_indirect_genes)


In [ ]:
hpv_positive_final_indirect.to_csv('Results/HPV Positive indirect results.csv', index=False)

In [ ]:
### extract the unique number of GENE_TARGETs from , seperated list in each of the GENE_TARGET column
def extract_unique_gene_targets(df, column_name):
    return len(set([gene.strip() for sublist in df[column_name].dropna().str.split(',') for gene in sublist]))

hpv_pos_final_indirect_genes = extract_unique_gene_targets(hpv_positive_final_indirect, 'GENE_TARGET')
print(hpv_pos_final_indirect_genes)

In [ ]:
hpv_positive_final_indirect

#### overall

In [ ]:
hpv_positive_final_direct

In [ ]:
extracted_target_df_combined[extracted_target_df_combined['GENE'] == 'PIK3CA']['PMID']

In [ ]:
### combined direct and indirect final results
### final columns: DRUG   GENE_TARGET CONNECTED_TO (risk gene)	NUM_DIRECT_TARGETS_HIT  
# Number of risk or immediate neighbor genes targeted	TOTAL_TARGETS_IN_DRUGBANK	PERCENTAGE_OF_TARGETS_HIT	
# ACTION	SPECIFIC_FUNCTION	drug_hypergeom_fdr	drug_empirical_fdr	
### column for Target description: direct, or indirect
hpv_positive_final_direct['Target_Description'] = 'Direct'
hpv_positive_final_indirect['Target_Description'] = 'Indirect'
hpv_positive_final_results = pd.concat([hpv_positive_final_direct, hpv_positive_final_indirect], ignore_index=True)
### replace any null with 'NA' in the hwole dataframe
hpv_positive_final_results = hpv_positive_final_results.fillna('NA')
hpv_positive_final_results = hpv_positive_final_results.groupby('DRUG').agg({
    'GENE_TARGET': lambda x: ', '.join(x),
    'CONNECTED_TO (risk gene)': lambda x: ', '.join(x),
    'NUM_DIRECT_TARGETS_HIT': 'first',
    'Number of risk or immediate neighbor genes targeted': 'first',
    'TOTAL_TARGETS_IN_DRUGBANK': 'first',
    'PERCENTAGE_OF_TARGETS_HIT': 'max',
    'ACTION': lambda x: ', '.join(x),
    'SPECIFIC_FUNCTION': lambda x: ', '.join(x),
    'drug_hypergeom_fdr': 'max',
    'drug_empirical_fdr': 'max',
    'Target_Description': lambda x: ', '.join(x)
}).reset_index()

In [ ]:
hpv_positive_final_results.sort_values(by = ['drug_empirical_fdr', 'PERCENTAGE_OF_TARGETS_HIT'], ascending = [ True, False ]).head(50)

In [ ]:
len(set(hpv_positive_final_results['DRUG']))

## HPV-

#### Genes

In [ ]:
### combine hpv negative somatic genes and cnv genes
hpv_negative_som_genes['MUT_TYPE'] = 'SOMATIC'
hpv_negative_som_genes['gene_name'] = hpv_negative_som_genes['Gene']
hpv_negative_som_genes['q_value']= hpv_negative_som_genes['Adjusted_P_Value']
hpv_negative_som_genes['empirical_q_value'] = hpv_negative_som_genes['Adjusted_Empirical_P_Value']
hpv_negative_combined_genes = pd.concat([hpv_negative_genes, hpv_negative_som_genes], axis=0)
### aggregate by GENE to get unique genes with both mutation types
hpv_negative_combined_genes['gene_name'] = hpv_negative_combined_genes['gene_name'].str.upper()
extracted_target_df_combined['GENE'] = extracted_target_df_combined['GENE'].str.upper()
hpv_negative_combined_genes = hpv_negative_combined_genes.groupby('gene_name').agg({
    'MUT_TYPE': lambda x: ', '.join(x),
    'q_value': lambda x: ', '.join(x.astype(str)) if len(x) > 1 else x.iloc[0].astype(str),
    'empirical_q_value': lambda x: ', '.join(x.astype(str)) if len(x) > 1 else x.iloc[0].astype(str)
}).reset_index()
# hpv_negative_combined_genes.sort_values(by='q_value')

In [ ]:
hpv_negative_combined_genes

In [ ]:
### merge genes with number of articles, pubmed id from literature data
hpv_negative_gene_results_with_lit = pd.merge(hpv_negative_combined_genes, extracted_target_df_combined, how = 'left', left_on='gene_name', right_on='GENE')
hpv_negative_gene_results_with_lit.drop(columns = ['INDEX'], inplace = True)

In [ ]:
hpv_negative_gene_results_with_lit = hpv_negative_gene_results_with_lit[hpv_negative_gene_results_with_lit['NUMBER_OF_ARTICLES']>0]

In [ ]:
hpv_negative_gene_results_with_lit.sort_values(by = ['empirical_q_value', 'q_value'], ascending=[True, True], inplace=True)

In [ ]:
hpv_negative_gene_results_with_lit.drop(columns=['INDEX', 'GENE'], errors='ignore', inplace=True)

In [ ]:
hpv_negative_gene_results_with_lit.to_csv('Results/HPV negative gene results.csv')

In [ ]:
hpv_negative_gene_results_with_lit

In [ ]:
len(set(hpv_negative_gene_results_with_lit['gene_name']))

In [ ]:
### export top genes to ouput final tables
hpv_negative_gene_results_with_lit.to_csv('Results/Final Results/HPV Negative validated genes.csv')

#### Direct

In [ ]:
### combine all hpv negative direct drug candidates
hpv_negative_final_direct = pd.concat([hpv_negative_direct_drug_candidates, hpv_negative_som_direct_drug_candidates])
### group by drug and comma seperate genes and mutation type
### columns: DRUG	GENE_TARGET	NUM_DIRECT_TARGETS_HIT	TOTAL_TARGETS_IN_DRUGBANK	PERCENTAGE_OF_TARGETS_HIT	GENE_GISTIC	GENE_normalized_gistic_score	
# ACTION	SPECIFIC_FUNCTION	drug_hypergeom_p_value      

hpv_negative_final_direct = hpv_negative_final_direct.groupby('DRUG').agg({'GENE_TARGET': lambda x: ', '.join(x),
                                               'MUT_TYPE': lambda x: ', '.join(x.unique()), ### unique mutation types per drug
                                                  'NUM_DIRECT_TARGETS_HIT': 'first',
                                                    'TOTAL_TARGETS_IN_DRUGBANK': 'first',
                                                    'PERCENTAGE_OF_TARGETS_HIT': 'first',
                                                    'ACTION': 'first',
                                                    'SPECIFIC_FUNCTION': 'first',
                                                    'drug_hypergeom_p_value': 'first',
                                                    'drug_hypergeom_fdr': 'first',
                                                    'drug_empirical_p_value': 'first',
                                                    'drug_empirical_fdr': 'first',
                                                    }).reset_index()

In [ ]:
extracted_target_df_combined[extracted_target_df_combined['GENE'] == 'EPHA2']

In [ ]:
### validate hpv negative direct drug candidates with literature data
hpv_negative_final_direct['PMID'] = ''
hpv_negative_final_direct['NUMBER_OF_ARTICLES'] = 0
hpv_negative_final_direct['LITERATURE_GENE_TARGETS'] = ''
for index, row in hpv_negative_final_direct.iterrows():
    gene_targets = row['GENE_TARGET'].split(', ')
    gene_targets = [gene.strip() for gene in gene_targets]
    gene_targets = list(set(gene_targets))
    pmids_set = set()
    literature_gene_targets = set()
    number_of_articles = 0
    for gene in gene_targets:
        matched_rows = extracted_target_df_combined[extracted_target_df_combined['GENE'] == gene]
        for _, matched_row in matched_rows.iterrows():
            pmids = matched_row['PMID'].split(', ')
            pmids_set.update(pmids)
            number_of_articles += matched_row['NUMBER_OF_ARTICLES']
            literature_gene_targets.add(matched_row['GENE'])
    
    hpv_negative_final_direct.at[index, 'LITERATURE_GENE_TARGETS'] = ', '.join(literature_gene_targets)
    hpv_negative_final_direct.at[index, 'PMID'] = ', '.join(pmids_set)
    hpv_negative_final_direct.at[index, 'NUMBER_OF_ARTICLES'] = number_of_articles

    

In [ ]:
hpv_negative_final_direct[hpv_negative_final_direct['NUMBER_OF_ARTICLES'] > 0].sort_values(by='NUMBER_OF_ARTICLES', ascending=False)

In [ ]:
### ensure that only drugs with NUMBER_OF_ARTICLES > 0 are saved, so that they have literature support
hpv_negative_final_direct= hpv_negative_final_direct[hpv_negative_final_direct['NUMBER_OF_ARTICLES'] > 0]
### save results
hpv_negative_final_direct.to_csv('Results/HPV Negative direct results.csv')

In [ ]:
hpv_negative_final_direct

#### Indirect

In [ ]:
hpv_negative_indirect_drug_candidates[hpv_negative_indirect_drug_candidates['DRUG'].str.lower()== 'quercetin']

In [ ]:
### combine all hpv negative indirect drug candidates
hpv_negative_final_indirect = pd.concat([hpv_negative_indirect_drug_candidates, hpv_negative_som_indirect_drug_candidates])
hpv_negative_final_indirect['ACTION'] = hpv_negative_final_indirect['ACTION'].fillna('UNKNOWN')
hpv_negative_final_indirect['SPECIFIC_FUNCTION'] = hpv_negative_final_indirect['SPECIFIC_FUNCTION'].fillna('UNKNOWN')
hpv_negative_final_indirect['drug_hypergeom_fdr'] = hpv_negative_final_indirect['drug_hypergeom_fdr'].fillna('UNKNOWN')
hpv_negative_final_indirect['drug_empirical_fdr'] = hpv_negative_final_indirect['drug_empirical_fdr'].fillna('UNKNOWN')
hpv_negative_final_indirect['MUT_TYPE'] = hpv_negative_final_indirect['MUT_TYPE'].fillna('UNKNOWN')

### group by drug and comma seperate genes and mutation type
### columns: DRUG	CONNECTED_TO (risk gene)
# Number of risk or immediate neighbor genes
# targeted	total_genes_targeted_in_drugbank  
# PERCENTAGE_OF_TARGETS_HIT
# Number of indirect genes connected to this risk gene  
# GENE_TARGET	
# GENE_Cohort_Frequency   
# GENE_Normalized_Count	
# GENE_Normalized_Cohort_Frequency

hpv_negative_final_indirect = hpv_negative_final_indirect.groupby (['DRUG']).agg({
    'GENE_TARGET': lambda x: ', '.join(x),
    'CONNECTED_TO (risk gene)': lambda x: ', '.join(x),
    'Number of risk or immediate neighbor genes targeted': 'first',
    'total_genes_targeted_in_drugbank': 'first',
    'PERCENTAGE_OF_TARGETS_HIT': 'first',
    'ACTION': lambda x: ', '.join(x),
    'SPECIFIC_FUNCTION': lambda x: ', '.join(x),
    'drug_hypergeom_fdr': 'max',
    'drug_empirical_fdr': 'max',
    'MUT_TYPE': lambda x: ', '.join(x)
}).reset_index()

In [ ]:
'artenimol' in hpv_negative_final_indirect['DRUG'].str.lower().values

In [ ]:
hpv_negative_final_indirect.sort_values(by = 'PERCENTAGE_OF_TARGETS_HIT', ascending = True).head(20)

In [ ]:
len(hpv_negative_final_indirect[hpv_negative_final_indirect['DRUG']=='fostamatinib']['GENE_TARGET'].iloc[0].split(','))

In [ ]:
### extract the unique number of GENE_TARGETs from , seperated list in each of the GENE_TARGET column
def extract_unique_gene_targets(df, column_name):
    return len(set([gene.strip() for sublist in df[column_name].dropna().str.split(',') for gene in sublist]))

hpv_negative_indirect_genes = extract_unique_gene_targets(hpv_negative_final_indirect, 'GENE_TARGET')
print(hpv_negative_indirect_genes)

In [ ]:
### add in literature validation columns
hpv_negative_final_indirect['PMID'] = ''
hpv_negative_final_indirect['NUMBER_OF_ARTICLES'] = 0
hpv_negative_final_indirect['LITERATURE_GENE_TARGETS'] = ''
for index, row in hpv_negative_final_indirect.iterrows():
    gene_targets = row['GENE_TARGET'].split(',')
    gene_targets = [gene.strip() for gene in gene_targets]
    pmids_set = set()
    literature_gene_targets = set()
    number_of_articles = 0
    for gene in gene_targets:
        matched_rows = extracted_target_df_combined[extracted_target_df_combined['GENE'] == gene]
        for _, matched_row in matched_rows.iterrows():
            pmids = matched_row['PMID'].split(',')
            pmids = [pmid.strip() for pmid in pmids]
            pmids_set.update(pmids)
            number_of_articles += matched_row['NUMBER_OF_ARTICLES']
            literature_gene_targets.add(matched_row['GENE'])
    
    hpv_negative_final_indirect.at[index, 'LITERATURE_GENE_TARGETS'] = ', '.join(literature_gene_targets)
    hpv_negative_final_indirect.at[index, 'PMID'] = ', '.join(pmids_set)
    hpv_negative_final_indirect.at[index, 'NUMBER_OF_ARTICLES'] = number_of_articles


### validate risk genes
hpv_negative_final_indirect = hpv_negative_final_indirect[hpv_negative_final_indirect['NUMBER_OF_ARTICLES'] > 0]
hpv_negative_final_indirect['RISK_GENE_PMID'] = ''
hpv_negative_final_indirect['RISK_GENE_NUMBER_OF_ARTICLES'] = 0
hpv_negative_final_indirect['RISK_GENE_LITERATURE_GENE_TARGETS'] = ''
for index, row in hpv_negative_final_indirect.iterrows():
    risk_genes = row['CONNECTED_TO (risk gene)'].split(',')
    risk_genes = [gene.strip() for gene in risk_genes]
    pmids_set = set()
    literature_gene_targets = set()
    number_of_articles = 0
    for gene in risk_genes:
        matched_rows = extracted_target_df_combined[extracted_target_df_combined['GENE'] == gene]
        for _, matched_row in matched_rows.iterrows():
            pmids = matched_row['PMID'].split(',')
            pmids = [pmid.strip() for pmid in pmids]
            pmids_set.update(pmids)
            number_of_articles += matched_row['NUMBER_OF_ARTICLES']
            literature_gene_targets.add(matched_row['GENE'])
    
    hpv_negative_final_indirect.at[index, 'RISK_GENE_LITERATURE_GENE_TARGETS'] = ', '.join(literature_gene_targets)
    hpv_negative_final_indirect.at[index, 'RISK_GENE_PMID'] = ', '.join(pmids_set)
    hpv_negative_final_indirect.at[index, 'RISK_GENE_NUMBER_OF_ARTICLES'] = number_of_articles

### make sure only drugs with literature support for both drug targets and risk genes are saved
hpv_negative_final_indirect = hpv_negative_final_indirect[hpv_negative_final_indirect['NUMBER_OF_ARTICLES'] > 0]
hpv_negative_final_indirect = hpv_negative_final_indirect[hpv_negative_final_indirect['RISK_GENE_NUMBER_OF_ARTICLES'] > 0]

hpv_negative_final_indirect.to_csv('Results/HPV Negative indirect results.csv', index=False)



In [ ]:
hpv_negative_final_indirect.sort_values(by = ['drug_empirical_fdr', 'PERCENTAGE_OF_TARGETS_HIT'], ascending=[True, False]).head(50)[['DRUG', 'drug_empirical_fdr','LITERATURE_GENE_TARGETS', 'RISK_GENE_PMID','RISK_GENE_LITERATURE_GENE_TARGETS','RISK_GENE_NUMBER_OF_ARTICLES','MUT_TYPE']]

#### overall

In [ ]:
hpv_negative_final_direct

In [ ]:
### combined direct and indirect final results
### final columns: DRUG   GENE_TARGET CONNECTED_TO (risk gene)	NUM_DIRECT_TARGETS_HIT  
# Number of risk or immediate neighbor genes targeted	TOTAL_TARGETS_IN_DRUGBANK	PERCENTAGE_OF_TARGETS_HIT	
# ACTION	SPECIFIC_FUNCTION	drug_hypergeom_fdr	drug_empirical_fdr	
### column for Target description: direct, or indirect
hpv_negative_final_direct['Target_Description'] = 'Direct'
hpv_negative_final_indirect['Target_Description'] = 'Indirect'
hpv_negative_final_results = pd.concat([hpv_negative_final_direct, hpv_negative_final_indirect], ignore_index=True)
### replace any null with 'NA' in the hwole dataframe
hpv_negative_final_results = hpv_negative_final_results.fillna('NA')
hpv_negative_final_results = hpv_negative_final_results.groupby('DRUG').agg({
    'GENE_TARGET': lambda x: ', '.join(x),
    'CONNECTED_TO (risk gene)': lambda x: ', '.join(x),
    'NUM_DIRECT_TARGETS_HIT': 'first',
    'Number of risk or immediate neighbor genes targeted': 'first',
    'TOTAL_TARGETS_IN_DRUGBANK': 'first',
    'PERCENTAGE_OF_TARGETS_HIT': 'max',
    'ACTION': lambda x: ', '.join(x),
    'SPECIFIC_FUNCTION': lambda x: ', '.join(x),
    'drug_hypergeom_fdr': 'max',
    'drug_empirical_fdr': 'max',
    'Target_Description': lambda x: ', '.join(x)
}).reset_index()

In [ ]:
len(set(hpv_negative_final_results['DRUG']))

In [ ]:
hpv_positive_final_direct

In [ ]:
hpv_positive_final_indirect

## Double check

In [ ]:
len(Drug_bank[Drug_bank['drug'].str.lower() == 'artenimol']['gene'].unique())
Drug_bank[Drug_bank['drug'].str.lower() == 'artenimol']['gene'].nunique()

In [ ]:
Drug_bank[(Drug_bank['drug'].str.lower() == 'quercetin') & (Drug_bank['gene'].str.lower() == 'stat3')]

In [ ]:
for gene in hpv_negative_final_indirect[hpv_negative_final_indirect['DRUG'].str.lower() == 'fostamatinib'.lower()]['LITERATURE_GENE_TARGETS']:
    print(gene)

In [ ]:
Drug_bank[(Drug_bank['drug'].str.lower() == 'quercetin'.lower()) & 
          (Drug_bank['gene'].str.lower() == 'stat3'.lower())]

In [ ]:
protein_interaction[(protein_interaction['combined_score']>700)&
                    (protein_interaction['Translated_protein_1'].str.lower() == 'stat3') & 
                    (protein_interaction['Translated_protein_2'].str.lower() == 'sox2')]